
# 🎛️ AI Cover End-to-End (RTX 4070, Jupyter)

**Pipeline:** Demucs (GPU) → *(optional)* torchcrepe F0 → RVC Inference *(bring-your-own model)* → Python Mixing → WAV/MP3 export

> ⚠️ Legal & ethical: Only use audio and voice models you have the right to use. No DRM breaking.


## 0) Environment check (CUDA + packages)

In [1]:

# If you already installed these in your aicover env, you can skip this cell.
# !pip install -q demucs librosa soundfile pedalboard pydub torchcrepe numpy==1.26.4

import torch, sys
print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda, "| Device:", torch.cuda.get_device_name(0))


Python: 3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:40:08) [MSC v.1938 64 bit (AMD64)]
CUDA available: True
CUDA: 12.1 | Device: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
import sys, torch
print(sys.executable)  # 应该是 ...\anaconda\envs\aicover\python.exe
print(torch.cuda.is_available(), torch.version.cuda)  # 应该是 True 12.1

C:\anaconda\envs\aicover\python.exe
True 12.1


## 1) Project folders

In [3]:

import os, pathlib
BASE = pathlib.Path("./ai-cover-project").resolve()
for sub in ["input","stems","vc_model","vc_output","mix","docs"]:
    (BASE/sub).mkdir(parents=True, exist_ok=True)
print("Project root:", BASE)


Project root: D:\change voice\ai-cover-project


## 2) Put your **legal** source audio in `ai-cover-project/input/`

In [6]:

# Option A: just place your WAV/MP3 in ./ai-cover-project/input/ and run this:
from pathlib import Path
import numpy as np, librosa, soundfile as sf

BASE = Path(r"D:\change voice\ai-cover-project").resolve()
input_dir = BASE / "input"

# 只找 wav/mp3，避免把 .ipynb_checkpoints 当文件
cands = [p for p in input_dir.iterdir() if p.is_file() and p.suffix.lower() in {".wav", ".mp3"}]
print("Found:", [p.name for p in cands])
assert cands, "input 里没有 WAV/MP3。"

src = cands[0]
y, sr = librosa.load(str(src), sr=44100, mono=False)
if y.ndim == 1:                      # 单声道→复制成双声道
    y = np.stack([y, y], axis=0)
y = y.T                              # (N, 2)

work_src = BASE / "input" / "source_stereo.wav"
sf.write(str(work_src), y, sr)
print("Stereo WAV ready:", work_src)



Found: ["Bryant Barnes,d4vd - I'd Rather Pretend.mp3", 'source.wav', 'source_stereo.wav']
Stereo WAV ready: D:\change voice\ai-cover-project\input\source_stereo.wav


## 3) Source separation with Demucs (GPU)

In [9]:

import os, sys, subprocess
from pathlib import Path

BASE = Path(r"D:\change voice\ai-cover-project").resolve()
os.environ["DEMUCS_CACHE_DIR"] = str(BASE / "demucs_cache")
work_src = BASE / "input" / "source_stereo.wav"

def run_demucs(device, model="mdx_extra"):
    cmd = [sys.executable, "-m", "demucs.separate",
           "-d", device, "--two-stems", "vocals",
           "-n", model, str(work_src), "-o", str(BASE / "stems")]
    print("\nRUN:", " ".join(cmd))
    p = subprocess.run(cmd, capture_output=True, text=True)
    print("=== STDOUT (tail) ===\n", p.stdout[-1200:])
    print("=== STDERR (tail) ===\n", p.stderr[-1200:])
    return p.returncode, model

rc, model = run_demucs("cuda", "mdx_extra")
if rc != 0:
    print("\n[Info] CUDA 失败，试 CPU 同模型…")
    rc, model = run_demucs("cpu", "mdx_extra")
if rc != 0:
    print("\n[Info] 继续试 htdemucs（不依赖 diffq）…")
    rc, model = run_demucs("cpu", "htdemucs")
assert rc == 0, "Demucs 仍失败，请把上面 STDERR 贴我。"

# === 关键：按模型名自动找结果目录 ===
candidates = [
    BASE / "stems" / work_src.stem,
    BASE / "stems" / model / work_src.stem,
    BASE / "stems" / "mdx_extra" / work_src.stem,
    BASE / "stems" / "htdemucs" / work_src.stem,
    BASE / "stems" / "mdx_q" / work_src.stem,
]
vocals_path = instrumental_path = None
root_used = None
for r in candidates:
    v = r / "vocals.wav"
    i = r / "no_vocals.wav"
    if v.exists() or i.exists():
        vocals_path, instrumental_path, root_used = v, i, r
        break

print("Result root:", root_used)
print("Vocals exists:", vocals_path and vocals_path.exists(), vocals_path)
print("Instrumental exists:", instrumental_path and instrumental_path.exists(), instrumental_path)



RUN: C:\anaconda\envs\aicover\python.exe -m demucs.separate -d cuda --two-stems vocals -n mdx_extra D:\change voice\ai-cover-project\input\source_stereo.wav -o D:\change voice\ai-cover-project\stems
=== STDOUT (tail) ===
 Selected model is a bag of 4 models. You will see that many progress bars per track.
Separated tracks will be stored in D:\change voice\ai-cover-project\stems\mdx_extra
Separating track D:\change voice\ai-cover-project\input\source_stereo.wav

=== STDERR (tail) ===
 ██████████████████████████████████████████████████████████████████| 198.0/198.0 [00:02<00:00, 75.55seconds/s]
100%|████████████████████████████████████████████████████████████████████████| 198.0/198.0 [00:02<00:00, 67.73seconds/s]

  0%|                                                                                  | 0.0/198.0 [00:00<?, ?seconds/s]
 17%|████████████▏                                                            | 33.0/198.0 [00:00<00:02, 62.46seconds/s]
 33%|████████████████████████▎      

In [11]:
from pathlib import Path

BASE = Path(r"D:\change voice\ai-cover-project").resolve()

# 优先用已存在的变量；否则自动在 stems 里搜最新一次分离结果
try:
    stems_root = Path(vocals_path).parent
    inst_path = Path(instrumental_path) if 'instrumental_path' in globals() else stems_root / "no_vocals.wav"
except NameError:
    candidates = sorted((BASE / "stems").rglob("vocals.wav"),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    assert candidates, "没找到分离结果：stems 里没有 vocals.wav。请先跑分离步骤。"
    vocals_path = candidates[0]
    stems_root = vocals_path.parent
    inst_path = stems_root / "no_vocals.wav"

print("stems_root:", stems_root)
print("vocals_path:", vocals_path)
print("inst_path:", inst_path, inst_path.exists())


stems_root: D:\change voice\ai-cover-project\stems\mdx_extra\source_stereo
vocals_path: D:\change voice\ai-cover-project\stems\mdx_extra\source_stereo\vocals.wav
inst_path: D:\change voice\ai-cover-project\stems\mdx_extra\source_stereo\no_vocals.wav True


## 3.1) (Optional) Work on a short 20s clip first (faster iteration)

In [12]:

# Create 20s clips for quick tests (does NOT need ffmpeg)
import soundfile as sf, librosa

def save_clip(src_path, out_path, sr=48000, seconds=20):
    y, _ = librosa.load(str(src_path), sr=sr, mono=True)
    max_n = min(len(y), seconds * sr)   # 防止歌曲不足20s时报错
    y = y[:max_n]
    sf.write(str(out_path), y, sr)

clip_dir = stems_root / "clips"
clip_dir.mkdir(parents=True, exist_ok=True)

vocals_clip = clip_dir / "vocals_20s.wav"
inst_clip   = clip_dir / "inst_20s.wav"

save_clip(vocals_path, vocals_clip)
save_clip(inst_path,  inst_clip)

print("Clips written:")
print("  vocals_clip:", vocals_clip, vocals_clip.exists())
print("  inst_clip:  ", inst_clip,   inst_clip.exists())



Clips written:
  vocals_clip: D:\change voice\ai-cover-project\stems\mdx_extra\source_stereo\clips\vocals_20s.wav True
  inst_clip:   D:\change voice\ai-cover-project\stems\mdx_extra\source_stereo\clips\inst_20s.wav True


## 4) (Optional) F0 extraction with **torchcrepe** (GPU)

In [13]:

import torch, torchcrepe, librosa, numpy as np
use_clip = True  # set False to use full vocals
src = vocals_clip if use_clip else vocals_path

y, sr = librosa.load(str(src), sr=16000, mono=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
audio = torch.tensor(y, device=device).unsqueeze(0)

f0 = torchcrepe.predict(
    audio, sr, hop_length=160,  # 10 ms
    fmin=50, fmax=1100,
    model='full', device=device, batch_size=2048,
    decoder=torchcrepe.decode.viterbi
)
np.save(str(BASE / "docs" / "f0.npy"), f0.squeeze().float().cpu().numpy())
tuple(f0.shape), f0[:5].flatten().tolist()


((1, 2001),
 [324.2936706542969,
  320.1251525878906,
  324.2769775390625,
  325.0162353515625,
  323.5555725097656,
  324.341064453125,
  324.3081359863281,
  324.1434020996094,
  323.60638427734375,
  325.880126953125,
  325.6902770996094,
  325.8323974609375,
  320.86920166015625,
  321.9012756347656,
  326.4090576171875,
  324.7839660644531,
  321.48309326171875,
  320.80072021484375,
  323.2754821777344,
  323.7825927734375,
  323.2139587402344,
  323.2738037109375,
  323.17913818359375,
  326.3335876464844,
  321.2291259765625,
  321.0476379394531,
  322.9010009765625,
  323.0810852050781,
  324.82379150390625,
  323.80377197265625,
  324.18695068359375,
  324.91485595703125,
  319.8811340332031,
  323.7720947265625,
  321.7574768066406,
  322.2643737792969,
  326.7271728515625,
  323.29901123046875,
  326.29278564453125,
  324.4757080078125,
  325.4158020019531,
  322.59210205078125,
  322.4408264160156,
  321.2281799316406,
  323.56915283203125,
  322.643798828125,
  323.292266


## 5) **RVC inference (BYO model & repo)**

- Place your `your_model.pth` (and optional `your_model.index`) into `ai-cover-project/vc_model/`
- Set the variables below.
- Then replace the PSEUDO command with your repo's actual inference command (from its README).
- If you paste your command to me, I’ll turn it into a one-click cell for you.


In [14]:

from pathlib import Path

# 路径（按需修改）
RVC_DIR     = Path(r"D:\5992\change voice\RVC1006Nvidia")                        # 你的 RVC 仓库目录
MODEL_PTH   = (BASE / "vc_model" / "XXXTENTACION.pth") # 你的 timbre 模型
MODEL_INDEX = (BASE / "vc_model" / "XXXTENTACION.index")  # 没有就设为 None
INPUT_VOX   = Path(vocals_path)                      # 用分离出的人声做转换
OUTPUT_VOX  = (BASE / "vc_output" / "vocals_converted.wav")

# 推理参数（常用甜点位）
F0_ALGO     = "crepe"   # 常见：crepe / rmvpe / harvest
INDEX_RATIO = 0.78      # 0.70~0.85 区间调
PITCH_SHIFT = 0         # 不换调，音域差大时 ±1/±2
DEVICE      = "cuda"    # 或 "cpu"

print("RVC_DIR exists:", RVC_DIR.exists())
print("MODEL_PTH exists:", MODEL_PTH.exists())
print("MODEL_INDEX exists:", (MODEL_INDEX and MODEL_INDEX.exists()))
print("INPUT_VOX:", INPUT_VOX)
print("OUTPUT_VOX:", OUTPUT_VOX)

# —— 提示：不同仓库命令不一样，下面给三种常见写法（选一，复制到下一格的 RVC_CMD 里改路径即可）——
print("\n⭐ 常见仓库命令模板（选其一，粘到下一格 RVC_CMD 里修改路径）：\n")

print("# A) WebUI 系的 CLI (示例名：infer_cli.py)：\n"
      f'python "{RVC_DIR}/infer_cli.py" '
      f'--model "{MODEL_PTH}" '
      f'--index "{MODEL_INDEX if MODEL_INDEX else ""}" '
      f'--input "{INPUT_VOX}" --out "{OUTPUT_VOX}" '
      f'--f0 {F0_ALGO} --index_ratio {INDEX_RATIO} --pitch {PITCH_SHIFT} --device {DEVICE}\n')

print("# B) 另一类仓库 (示例名：infer.py)：\n"
      f'python "{RVC_DIR}/infer.py" '
      f'--model "{MODEL_PTH}" '
      f'--index "{MODEL_INDEX if MODEL_INDEX else ""}" '
      f'--input "{INPUT_VOX}" --out "{OUTPUT_VOX}" '
      f'--f0 {F0_ALGO} --index_ratio {INDEX_RATIO} --pitch {PITCH_SHIFT} --device {DEVICE}\n')

print("# C) 有些仓库叫 inference_main.py：\n"
      f'python "{RVC_DIR}/inference_main.py" '
      f'-m "{MODEL_PTH}" '
      f'--index "{MODEL_INDEX if MODEL_INDEX else ""}" '
      f'-i "{INPUT_VOX}" -o "{OUTPUT_VOX}" '
      f'--f0 {F0_ALGO} --ir {INDEX_RATIO} --pitch {PITCH_SHIFT} --device {DEVICE}\n')



RVC_DIR exists: True
MODEL_PTH exists: True
MODEL_INDEX exists: True
INPUT_VOX: D:\change voice\ai-cover-project\stems\mdx_extra\source_stereo\vocals.wav
OUTPUT_VOX: D:\change voice\ai-cover-project\vc_output\vocals_converted.wav

⭐ 常见仓库命令模板（选其一，粘到下一格 RVC_CMD 里修改路径）：

# A) WebUI 系的 CLI (示例名：infer_cli.py)：
python "D:\5992\change voice\RVC1006Nvidia/infer_cli.py" --model "D:\change voice\ai-cover-project\vc_model\XXXTENTACION.pth" --index "D:\change voice\ai-cover-project\vc_model\XXXTENTACION.index" --input "D:\change voice\ai-cover-project\stems\mdx_extra\source_stereo\vocals.wav" --out "D:\change voice\ai-cover-project\vc_output\vocals_converted.wav" --f0 crepe --index_ratio 0.78 --pitch 0 --device cuda

# B) 另一类仓库 (示例名：infer.py)：
python "D:\5992\change voice\RVC1006Nvidia/infer.py" --model "D:\change voice\ai-cover-project\vc_model\XXXTENTACION.pth" --index "D:\change voice\ai-cover-project\vc_model\XXXTENTACION.index" --input "D:\change voice\ai-cover-project\stems\mdx_extra\source_

In [19]:

from pathlib import Path
import subprocess
RVC_DIR = Path(r"D:\5992\change voice\RVC1006Nvidia")  # ← 改成你的仓库根
RVC_PY  = RVC_DIR / "runtime" / "python.exe"
assert RVC_PY.exists(), f"找不到 runtime Python: {RVC_PY}"

# 直接强制重装兼容组合（Py3.9 → numpy 1.26.4, llvmlite 0.41.1, numba 0.58.1）
cmd = [str(RVC_PY), "-m", "pip", "install", "--force-reinstall", "--no-cache-dir",
       "numpy==1.26.4", "llvmlite==0.41.1", "numba==0.58.1"]
print("RUN:", " ".join(cmd))
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-2000:])
print("\n--- STDERR (tail) ---\n", p.stderr[-2000:])



RUN: D:\5992\change voice\RVC1006Nvidia\runtime\python.exe -m pip install --force-reinstall --no-cache-dir numpy==1.26.4 llvmlite==0.41.1 numba==0.58.1
--------------- 1.0/2.6 MB 6.9 MB/s eta 0:00:01
   --------------- ------------------------ 1.0/2.6 MB 6.6 MB/s eta 0:00:01
   ------------------- -------------------- 1.3/2.6 MB 5.3 MB/s eta 0:00:01
   ------------------------ --------------- 1.6/2.6 MB 5.6 MB/s eta 0:00:01
   ---------------------------- ----------- 1.9/2.6 MB 5.7 MB/s eta 0:00:01
   ------------------------------- -------- 2.1/2.6 MB 5.8 MB/s eta 0:00:01
   ------------------------------- -------- 2.1/2.6 MB 5.8 MB/s eta 0:00:01
   ------------------------------- -------- 2.1/2.6 MB 5.8 MB/s eta 0:00:01
   ------------------------------- -------- 2.1/2.6 MB 5.8 MB/s eta 0:00:01
   ------------------------------- -------- 2.1/2.6 MB 5.8 MB/s eta 0:00:01
   ------------------------------- -------- 2.1/2.6 MB 5.8 MB/s eta 0:00:01
   ------------------------------- -----

In [21]:
from pathlib import Path
import subprocess
RVC_DIR = Path(r"D:\5992\change voice\RVC1006Nvidia")
RVC_PY  = RVC_DIR / "runtime" / "python.exe"

# 你之前缺过 dotenv；顺手补 soundfile/scipy
for pkgs in [["python-dotenv"], ["soundfile","scipy"]]:
    subprocess.run([str(RVC_PY), "-m", "pip", "install", "-U", *pkgs], check=False)

# 自检（应打印 ok + 版本号）
test = subprocess.run([str(RVC_PY), "-c",
                       "import numpy, numba, librosa; "
                       "print('ok', numpy.__version__, numba.__version__)"],
                      capture_output=True, text=True)
print(test.stdout or test.stderr)


ok 1.26.4 0.58.1



In [31]:
# === 自动执行 RVC 推理（尝试常见脚本/参数风格） ===
from pathlib import Path
import shutil

# 路径按你的实际改
BASE     = Path(r"D:\change voice\ai-cover-project").resolve()
RVC_DIR  = Path(r"D:\5992\change voice\RVC1006Nvidia").resolve()

MODEL_PTH   = BASE / "vc_model" / "XXXTENTACION.pth"     # 你的模型
MODEL_INDEX = BASE / "vc_model" / "XXXTENTACION.index"   # 可选

# 复制到仓库的 assets/weights/
weights_dir = RVC_DIR / "assets" / "weights"
weights_dir.mkdir(parents=True, exist_ok=True)
dst_model = weights_dir / MODEL_PTH.name
if not dst_model.exists():
    shutil.copy2(MODEL_PTH, dst_model)
print("Model placed at:", dst_model)

# 记录下来：后面推理只传“文件名”
MODEL_NAME_FOR_CLI = MODEL_PTH.name
print("Use model_name:", MODEL_NAME_FOR_CLI)




Model placed at: D:\5992\change voice\RVC1006Nvidia\assets\weights\XXXTENTACION.pth
Use model_name: XXXTENTACION.pth


In [30]:
import sys, subprocess
print("PY:", sys.executable)  # 确认是 ...\envs\aicover\python.exe
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "av"])
# 自检
import av, torch
print("PyAV:", av.__version__)
print("CUDA:", torch.cuda.is_available(), torch.version.cuda)


PY: C:\anaconda\envs\aicover\python.exe
PyAV: 15.0.0
CUDA: True 12.1


In [33]:
import sys, subprocess
print("PY:", sys.executable)  # 确认是 ...\envs\aicover\python.exe
# 方式 1：pip（多数 Windows/Py39 能装上）
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "faiss-cpu"])
# 验证
import faiss, numpy as np
print("faiss ok:", faiss.__version__)


PY: C:\anaconda\envs\aicover\python.exe
faiss ok: 1.11.0


In [35]:
import sys, subprocess
print("PY:", sys.executable)  # 确认是 ...\envs\aicover\python.exe
pkgs = [
    "praat-parselmouth",  # parselmouth
    "faiss-cpu",          # 用到了 --index_path / --index_rate 就必须有
    "pyworld",            # 某些管线会用到
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *pkgs])

# 简单自检
import parselmouth, faiss, pyworld
print("OK:", parselmouth.__version__)
print("faiss:", faiss.__version__)


PY: C:\anaconda\envs\aicover\python.exe
OK: 0.4.6
faiss: 1.11.0


In [42]:
from pathlib import sys
from pathlib import Path   # Path 类来自 pathlib
import subprocess   
# 你的 RVC 仓库根（确认没写错空格/路径）
RVC_DIR = Path(r"D:\5992\change voice\RVC1006Nvidia").resolve()
RVC_PY  = RVC_DIR / "runtime" / "python.exe"
assert RVC_PY.exists(), f"找不到 runtime Python: {RVC_PY}"
print("Using runtime:", RVC_PY)

# 1) 安装 CUDA 版 PyTorch（cu121）到 runtime
subprocess.check_call([str(RVC_PY), "-m", "pip", "install", "-U",
                       "torch", "torchvision", "torchaudio",
                       "--index-url", "https://download.pytorch.org/whl/cu121"])

# 2) 核心依赖（之前报错过的都补上）
basic_pkgs = [
    "python-dotenv", "soundfile", "scipy", "praat-parselmouth",
    "pyworld", "faiss-cpu", "av"
]
subprocess.check_call([str(RVC_PY), "-m", "pip", "install", "-U", *basic_pkgs])

# 3) fairseq（优先用已打好的 Windows cp39 轮子）
#    若网络取不下来，可以把下面 URL 换成你手工下载到本地的 whl 路径
FAIRSEQ_WHL = "https://huggingface.co/Jmica/rvc/resolve/main/fairseq-0.12.2-cp39-cp39-win_amd64.whl"  # cp39专用
try:
    subprocess.check_call([str(RVC_PY), "-m", "pip", "install", FAIRSEQ_WHL])
except Exception as e:
    print("[Warn] 在线装 fairseq 失败：", e, "\n→ 你也可以手动下载该 whl 后本地安装：  runtime\\python.exe -m pip install D:\\pkgs\\fairseq-0.12.2-cp39-cp39-win_amd64.whl")

# 4) 快速自检：CUDA & 关键包
check = subprocess.run([str(RVC_PY), "-c",
                        "import torch,sys;print('CUDA',torch.cuda.is_available(),torch.version.cuda);"
                        "import fairseq,librosa,parselmouth,faiss,av;print('OK fairseq')"],
                       capture_output=True, text=True)
print(check.stdout or check.stderr)


Using runtime: D:\5992\change voice\RVC1006Nvidia\runtime\python.exe
CUDA False None



In [43]:
from pathlib import Path   # Path 类来自 pathlib
import subprocess 

RVC_DIR = Path(r"D:\5992\change voice\RVC1006Nvidia").resolve()
RVC_PY  = RVC_DIR / "runtime" / "python.exe"
assert RVC_PY.exists(), RVC_PY

# 1) 卸载 CPU 版
subprocess.run([str(RVC_PY), "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], check=False)

# 2) 安装 CUDA 版（固定一个稳定组合）
cmd = [str(RVC_PY), "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
       "--index-url", "https://download.pytorch.org/whl/cu121",
       "torch==2.3.1+cu121", "torchvision==0.18.1+cu121", "torchaudio==2.3.1+cu121"]
print("RUN:", " ".join(cmd))
subprocess.check_call(cmd)

# 3) 验证：应看到 True, 12.1, 你的GPU名
test = subprocess.run([str(RVC_PY), "-c",
                       "import torch; "
                       "print('cuda?', torch.cuda.is_available(), 'cuda', torch.version.cuda); "
                       "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')"],
                      capture_output=True, text=True)
print(test.stdout or test.stderr)


RUN: D:\5992\change voice\RVC1006Nvidia\runtime\python.exe -m pip install --no-cache-dir --force-reinstall --index-url https://download.pytorch.org/whl/cu121 torch==2.3.1+cu121 torchvision==0.18.1+cu121 torchaudio==2.3.1+cu121
cuda? True cuda 12.1
NVIDIA GeForce RTX 4060 Laptop GPU



In [45]:
from pathlib import Path   # Path 类来自 pathlib
import subprocess 

RVC_DIR = Path(r"D:\5992\change voice\RVC1006Nvidia")
RVC_PY  = RVC_DIR / "runtime" / "python.exe"

# 已经装过会跳过/快速通过
subprocess.check_call([str(RVC_PY), "-m", "pip", "install", "-U",
    "python-dotenv", "soundfile", "scipy", "praat-parselmouth", "pyworld", "av"
])


0

In [48]:
from pathlib import Path   # Path 类来自 pathlib
import subprocess 

RVC_DIR = Path(r"D:\5992\change voice\RVC1006Nvidia")
RVC_PY  = RVC_DIR / "runtime" / "python.exe"
assert RVC_PY.exists(), RVC_PY

# 1) 强制用稳定组合（NumPy 1.26.4 + SciPy 1.10.1）
subprocess.run([str(RVC_PY), "-m", "pip", "uninstall", "-y", "scipy"], check=False)
subprocess.check_call([str(RVC_PY), "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
                       "numpy==1.26.4"])
subprocess.check_call([str(RVC_PY), "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
                       "--only-binary=:all:",  # 避免源码编译
                       "scipy==1.10.1"])

# 2) 快速自检：这些 import 都要能过
test = subprocess.run([str(RVC_PY), "-c",
                       "import numpy,scipy,librosa; "
                       "from scipy import interpolate, signal; "
                       "print('OK', numpy.__version__, scipy.__version__)"],
                      capture_output=True, text=True)
print(test.stdout or test.stderr)


OK 1.26.4 1.10.1



In [50]:
from pathlib import Path
import re, shutil

RVC_DIR = Path(r"D:\5992\change voice\RVC1006Nvidia").resolve()
audio_py = RVC_DIR / "infer" / "lib" / "audio.py"
assert audio_py.exists(), f"找不到 {audio_py}"

# 备份一份
bak = audio_py.with_suffix(".py.bak")
if not bak.exists():
    shutil.copy2(audio_py, bak)

txt = audio_py.read_text(encoding="utf-8")
new = re.sub(r'av\.open\(([^,]+),\s*["\']rb["\']\)', r'av.open(\1, "r")', txt)
new = re.sub(r'av\.open\(([^,]+),\s*["\']wb["\']\)\)', r'av.open(\1, "w"))', new)  # 若代码里有写模式也一并修

if txt != new:
    audio_py.write_text(new, encoding="utf-8")
    print("Patched:", audio_py)
else:
    print("No 'rb'/'wb' pattern found,可能已修过。文件未改动。")


Patched: D:\5992\change voice\RVC1006Nvidia\infer\lib\audio.py


In [51]:
from pathlib import Path, os

out_wav = Path(r"D:\change voice\ai-cover-project\vc_output\vocals_rvc.wav")
if out_wav.exists():
    try:
        size = out_wav.stat().st_size
        if size < 1024:  # 小于1KB基本就是坏文件
            out_wav.unlink()
            print("Removed tiny/invalid file:", out_wav)
        else:
            print("Existing output seems OK (size bytes):", size)
    except Exception as e:
        print("Check/remove failed:", e)
else:
    print("No existing output. Good to go.")


Removed tiny/invalid file: D:\change voice\ai-cover-project\vc_output\vocals_rvc.wav


In [1]:
from pathlib import Path   # Path 在 pathlib 里
import re                  # re 是正则表达式模块
import shutil   

RVC_DIR = Path(r"D:\5992\change voice\RVC1006Nvidia").resolve()
audio_py = RVC_DIR / "infer" / "lib" / "audio.py"
assert audio_py.exists(), f"找不到 {audio_py}"

# 备份
bak = audio_py.with_suffix(".py.bak")
if not bak.exists():
    shutil.copy2(audio_py, bak)

txt = audio_py.read_text(encoding="utf-8")

# 将 av.open(..., "rb" [,...]) -> "r"
txt2 = re.sub(r'(av\.open\([^,]+,\s*)[\'"]rb[\'"]', r'\1"r"', txt)
# 将 av.open(..., "wb" [,...]) -> "w"
txt2 = re.sub(r'(av\.open\([^,]+,\s*)[\'"]wb[\'"]', r'\1"w"', txt2)

if txt2 != txt:
    audio_py.write_text(txt2, encoding="utf-8")
    print("Patched:", audio_py)
else:
    print("未发现需要替换的 rb/wb（可能已修过）")


Patched: D:\5992\change voice\RVC1006Nvidia\infer\lib\audio.py


In [2]:
from pathlib import Path
out_wav = Path(r"D:\change voice\ai-cover-project\vc_output\vocals_rvc.wav")
if out_wav.exists() and out_wav.stat().st_size == 0:
    out_wav.unlink()
    print("Removed empty file:", out_wav)
else:
    print("No empty file to remove or file has content.")


Removed empty file: D:\change voice\ai-cover-project\vc_output\vocals_rvc.wav


In [4]:
from pathlib import Path
import re, shutil

RVC_DIR = Path(r"D:\5992\change voice\RVC1006Nvidia").resolve()
audio_py = RVC_DIR / "infer" / "lib" / "audio.py"
assert audio_py.exists(), f"找不到 {audio_py}"

# 先备份
bak = audio_py.with_suffix(".py.bak2")
if not bak.exists():
    shutil.copy2(audio_py, bak)

src = audio_py.read_text(encoding="utf-8")

# 1) 用纯 librosa/soundfile 的实现替换 load_audio
load_audio_impl = r'''
def load_audio(path, sr):
    """
    读取任意音频，转单声道并重采样到 sr，返回 float32 [-1,1] 的一维 numpy 数组。
    不依赖 PyAV，避免 rb/wb 模式错误。
    """
    import numpy as np
    import soundfile as sf
    import librosa

    # 尝试 librosa 统一读取 + 重采样
    try:
        y, _ = librosa.load(path, sr=sr, mono=True)
        return y.astype(np.float32)
    except Exception:
        # 回退到 soundfile，然后必要时再用 librosa 重采样
        data, fr = sf.read(path, always_2d=False)
        if data.ndim > 1:
            data = data.mean(axis=1)
        data = data.astype(np.float32)
        if fr != sr:
            data = librosa.resample(data, orig_sr=fr, target_sr=sr)
        return data.astype(np.float32)
'''

# 2) （可选）保留一个空壳 audio2，防止其他地方引用
audio2_stub = r'''
def audio2(*args, **kwargs):
    # 兼容旧代码：不再用 PyAV 转码，直接在 load_audio 里完成读取与重采样。
    raise NotImplementedError("audio2 is not used; load_audio handles decoding/resample.")
'''

# 用正则替换 load_audio 定义
src2 = re.sub(r'\ndef\s+load_audio\s*\(.*?\)\s*:[\s\S]*?(?=\ndef\s|\Z)', '\n' + load_audio_impl + '\n', src, flags=re.DOTALL)

# 若存在 audio2 定义，也替换为 stub（否则追加一个）
if re.search(r'\ndef\s+audio2\s*\(.*?\)\s*:', src2):
    src2 = re.sub(r'\ndef\s+audio2\s*\(.*?\)\s*:[\s\S]*?(?=\ndef\s|\Z)', '\n' + audio2_stub + '\n', src2, flags=re.DOTALL)
else:
    src2 += '\n' + audio2_stub + '\n'

if src2 != src:
    audio_py.write_text(src2, encoding="utf-8")
    print("Patched:", audio_py)
else:
    print("文件未改动（可能已打过补丁）。")


Patched: D:\5992\change voice\RVC1006Nvidia\infer\lib\audio.py


In [7]:
from pathlib import Path
out_wav = Path(r"D:\change voice\ai-cover-project\vc_output\vocals_rvc.wav")
if out_wav.exists() and out_wav.stat().st_size == 0:
    out_wav.unlink()
    print("Removed empty file:", out_wav)
else:
    print("Output not present or already non-empty.")


Removed empty file: D:\change voice\ai-cover-project\vc_output\vocals_rvc.wav


In [8]:
from pathlib import Path
p = Path(r"D:\change voice\ai-cover-project\vc_output\vocals_rvc.wav")
if p.exists() and p.stat().st_size == 0:
    p.unlink()

In [12]:
import os, subprocess
from pathlib import Path

BASE    = Path(r"D:\change voice\ai-cover-project").resolve()
RVC_DIR = Path(r"D:\5992\change voice\RVC1006Nvidia").resolve()
PY      = RVC_DIR / "runtime" / "python.exe"
script  = RVC_DIR / "tools" / "infer_cli.py"

# 输入/输出
vocals_path = sorted((BASE / "stems").rglob("vocals.wav"),
                     key=lambda p: p.stat().st_mtime, reverse=True)[0]
OUTPUT_DIR  = BASE / "vc_output"; OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VOX  = OUTPUT_DIR / (Path(vocals_path).stem + "_rvc.wav")

env = os.environ.copy()
env["PYTHONPATH"] = str(RVC_DIR) + os.pathsep + env.get("PYTHONPATH", "")

cmd = [str(PY), str(script),
       "--input_path", str(vocals_path),
       "--opt_path",   str(OUTPUT_VOX),
       "--model_name", "XXXTENTACION.pth",  # 模型文件名（放在 assets/weights/）
       "--index_rate", "0",                  # 索引先关掉
       "--f0method",   "crepe",
       "--f0up_key",   "0",
       "--device",     "cuda",
       "--index_path", " " ]

print("RUN:", " ".join(cmd))
p = subprocess.run(cmd, cwd=str(RVC_DIR), env=env, capture_output=True, text=True)
print("=== STDOUT (tail) ===\n", p.stdout[-1200:])
print("=== STDERR (tail) ===\n", p.stderr[-1200:])
print("OK?", p.returncode, OUTPUT_VOX.exists(), OUTPUT_VOX, "size:",
      OUTPUT_VOX.stat().st_size if OUTPUT_VOX.exists() else 0)





RUN: D:\5992\change voice\RVC1006Nvidia\runtime\python.exe D:\5992\change voice\RVC1006Nvidia\tools\infer_cli.py --input_path D:\change voice\ai-cover-project\stems\mdx_extra\source_stereo\vocals.wav --opt_path D:\change voice\ai-cover-project\vc_output\vocals_rvc.wav --model_name XXXTENTACION.pth --index_rate 0 --f0method crepe --f0up_key 0 --device cuda --index_path  
=== STDOUT (tail) ===
 is_half:True, device:cuda:0

=== STDERR (tail) ===
 h
2025-08-12 17:34:37 | INFO | infer.modules.vc.modules | Loading: assets/weights/XXXTENTACION.pth
D:\5992\change voice\RVC1006Nvidia\runtime\lib\site-packages\torch\nn\utils\weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
2025-08-12 17:34:40 | INFO | infer.modules.vc.modules | Select index: 
D:\5992\change voice\RVC1006Nvidia\runtime\lib\site-package

## 6) Mixing in Python (EQ/De-Esser/Comp/Reverb/Limiter)

In [6]:
!pip show pedalboard
!pip install -U "pedalboard>=0.9.7"



  Using cached pedalboard-0.9.17-cp312-cp312-win_amd64.whl.metadata (58 kB)
Using cached pedalboard-0.9.17-cp312-cp312-win_amd64.whl (3.6 MB)


In [2]:
import sys, subprocess
print("Installing into:", sys.executable)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pedalboard==0.9.17", "librosa", "soundfile"])


Installing into: C:\anaconda\envs\aicover\python.exe


0

In [5]:
# === 初始化：自动设置 BASE / vocals_path / inst_path / OUTPUT_VOX ===
from pathlib import Path

# 1) 可能的项目根目录（按顺序尝试；不对就手动改成你的路径）
CAND_BASES = [
    Path(r"D:\change voice\ai-cover-project"),
    Path(r"D:\change voive\ai-cover-project"),
]
BASE = next((p for p in CAND_BASES if p.exists()), None)
assert BASE is not None, f"找不到项目根目录，请把 CAND_BASES 改成你的真实路径，例如 D:\\change voice\\ai-cover-project"
print("BASE =", BASE)

# 2) 找到最新的 Demucs 人声与伴奏
vocals_cands = sorted(BASE.rglob(r"stems/**/vocals.wav"), key=lambda p: p.stat().st_mtime)
assert vocals_cands, "找不到 Demucs 导出的人声 vocals.wav，请确认已完成分离步骤。"
vocals_path = vocals_cands[-1]

# 尝试匹配伴奏文件名（不同模型命名可能不同）
inst_path = vocals_path.parent / "no_vocals.wav"
if not inst_path.exists():
    for alt in ("accompaniment.wav", "instrumental.wav", "other.wav"):
        if (vocals_path.parent / alt).exists():
            inst_path = vocals_path.parent / alt
            break
assert inst_path.exists(), f"找不到伴奏：{inst_path}（或同目录下的 accompaniment/instrumental/other.wav）"

# 3) 确定 RVC 输出（若不存在则给出预期路径，混音代码会自动兜底）
vc_dir = BASE / "vc_output"
vc_dir.mkdir(parents=True, exist_ok=True)
rvc_outs = sorted(vc_dir.glob("*_rvc.wav"), key=lambda p: p.stat().st_mtime)
OUTPUT_VOX = rvc_outs[-1] if rvc_outs else (vc_dir / (vocals_path.stem + "_rvc.wav"))

# 4) song_path（用于 clips 命名的 stem；没有就用 vocals 上级名）
song_path = BASE / "input" / "source_stereo.wav"
if not song_path.exists():
    song_path = vocals_path.parent  # 仅用来拿 stem，不一定是实际文件
print("[OK] 变量就绪：")
print("  vocals_path :", vocals_path)
print("  inst_path   :", inst_path)
print("  OUTPUT_VOX  :", OUTPUT_VOX, "(exists:", OUTPUT_VOX.exists(), ")")
print("  song_stem   :", song_path.stem)


BASE = D:\change voice\ai-cover-project
[OK] 变量就绪：
  vocals_path : D:\change voice\ai-cover-project\stems\mdx_extra\source_stereo\vocals.wav
  inst_path   : D:\change voice\ai-cover-project\stems\mdx_extra\source_stereo\no_vocals.wav
  OUTPUT_VOX  : D:\change voice\ai-cover-project\vc_output\vocals_rvc.wav (exists: True )
  song_stem   : source_stereo


In [7]:
import os, sys, librosa, soundfile as sf
import numpy as np
from pathlib import Path
from pedalboard import Pedalboard, HighpassFilter, Compressor, Gain, Reverb, Limiter
# ↓ 有 Deesser 就用；没有就降级为峰值/高搁架滤波；都没有就跳过
try:
    from pedalboard import Deesser
    _deess = Deesser()
except Exception:
    _deess = None
    try:
        from pedalboard import PeakFilter
        _deess = PeakFilter(frequency_hz=7500.0, gain_db=-4.5, q=1.2)
    except Exception:
        try:
            from pedalboard import HighShelfFilter
            _deess = HighShelfFilter(cutoff_frequency_hz=7000.0, gain_db=-4.0)
        except Exception:
            _deess = Gain(gain_db=0.0)  # 最后兜底：不做处理

from pedalboard.io import AudioFile

# ---- 基础路径与变量（这些变量应该已在前面格里设置过）----
assert 'BASE' in globals(), "缺少 BASE 变量"
BASE = Path(BASE)

# 上一步 RVC 的输出（如果你在 runtime 推理，文件就在 BASE/vc_output/ 下）
cand = Path(OUTPUT_VOX) if 'OUTPUT_VOX' in globals() else BASE / "vc_output" / "vocals_rvc.wav"

def nonempty(p: Path, min_bytes=2048):
    return p.exists() and p.stat().st_size >= min_bytes

assert 'vocals_path' in globals() and 'inst_path' in globals(), "缺少 vocals_path / inst_path 变量"

# 如果存在 clip 变量就用，否则忽略
song_stem = (globals().get('song_path', Path('source_stereo.wav'))).stem
vocals_clip = BASE / "stems" / song_stem / "clips" / "vocals_20s.wav"
inst_clip   = BASE / "stems" / song_stem / "clips" / "inst_20s.wav"

if nonempty(cand):
    vox_in = cand
elif vocals_clip.exists():
    vox_in = vocals_clip
else:
    vox_in = Path(vocals_path)

inst_in = inst_clip if inst_clip.exists() else Path(inst_path)

print("[Files]")
print("  Vocal in :", vox_in, vox_in.exists(), vox_in.stat().st_size if vox_in.exists() else 0)
print("  Inst  in :", inst_in, inst_in.exists(), inst_in.stat().st_size if inst_in.exists() else 0)

# ---- 读取并统一到目标采样率 ----
TARGET_SR = 48000
inst, _ = librosa.load(str(inst_in), sr=TARGET_SR, mono=False)
vox,  _ = librosa.load(str(vox_in),  sr=TARGET_SR, mono=False)

# ✅ 统一转成 (channels, n_samples)
def to_ch_first(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y)
    if y.ndim == 1:                      # 单声道 → 复制为 2ch
        return np.vstack([y, y])
    # y.ndim == 2
    # librosa(mono=False) 常见是 (n, ch)；如果 time 轴更长，就转置
    return y.T if y.shape[0] > y.shape[1] else y

inst = to_ch_first(inst)
vox  = to_ch_first(vox)

# 对齐长度
L = min(inst.shape[-1], vox.shape[-1])
inst = inst[:, :L]
vox  = vox[:,  :L]

# （可选）打印检查，避免再次维度错误
print("inst shape:", inst.shape)  # 期望 (2, n)
print("vox  shape:", vox.shape)   # 期望 (2, n)

# ---- 简单对齐电平 ----
def rms(x): 
    return np.sqrt(np.mean(np.square(x), axis=-1, keepdims=True) + 1e-12)
inst_rms = np.mean(rms(inst))
vox_rms  = np.mean(rms(vox))
target_rms = max(inst_rms, 1e-4)
if vox_rms > 0:
    vox = vox * float(target_rms / vox_rms * 0.9)

# ---- 插件链并逐通道处理 ----
chain = Pedalboard([
    HighpassFilter(cutoff_frequency_hz=80.0),
    _deess,
    Compressor(threshold_db=-18, ratio=2.5),
    Reverb(room_size=0.12, wet_level=0.08, dry_level=0.92),
    Gain(gain_db=0.0),
])

# 逐通道处理，保持形状 (2, n)
vox_proc = np.vstack([chain(vox[ch], TARGET_SR) for ch in range(vox.shape[0])])

# ✅ 现在两者形状一致，可安全相加
mix = inst + 0.9 * vox_proc

# 峰值限幅
mix = np.vstack([Pedalboard([Limiter(threshold_db=-1.0)])(mix[ch], TARGET_SR)
                 for ch in range(mix.shape[0])])

# 防止溢出
mx = np.max(np.abs(mix))
if mx > 0.999:
    mix = mix / mx * 0.999

# ---- 导出 ----
outname = "final_mix_clip.wav" if vox_in == vocals_clip else "final_mix.wav"
final_wav = BASE / "mix" / outname
final_wav.parent.mkdir(parents=True, exist_ok=True)

# AudioFile 需要 [n_samples, channels]
with AudioFile(str(final_wav), 'w', TARGET_SR, mix.shape[0]) as f:
    f.write(mix.T.astype(np.float32))

print("✅ Done:", final_wav, "| size:", final_wav.stat().st_size)
str(final_wav)



[Files]
  Vocal in : D:\change voice\ai-cover-project\vc_output\vocals_rvc.wav True 14761644
  Inst  in : D:\change voice\ai-cover-project\stems\mdx_extra\source_stereo\no_vocals.wav True 32556428
inst shape: (2, 8856960)
vox  shape: (2, 8856960)
✅ Done: D:\change voice\ai-cover-project\mix\final_mix.wav | size: 35427944


'D:\\change voice\\ai-cover-project\\mix\\final_mix.wav'

## 7) (Optional) Export MP3 (needs ffmpeg in PATH)

In [ ]:

from pydub import AudioSegment
from shutil import which

ff = which("ffmpeg") or which("ffmpeg.exe")
if ff is None:
    print("ffmpeg not found in PATH — skipping MP3 export. Install ffmpeg or export WAV only.")
else:
    mp3_out = str(final_wav).replace(".wav", ".mp3")
    AudioSegment.from_wav(str(final_wav)).export(mp3_out, format="mp3", bitrate="320k")
    mp3_out
